In [10]:
%load_ext autoreload
%autoreload 2   

In [11]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) 

In [12]:
import sys
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))

from collections import OrderedDict
from pathlib import Path

import representation_spaces as rs
from data import GoEmotions
import part1
import torch


## Data statistics

#TODO: write dataset description

#TODO: how to handle multi-class data points?

#TODO: merge dev and train for cross-validation?

In [13]:
ds = GoEmotions(drop_multi_labels=True)
ds.summary

In [14]:
ds.plot_label_dist()

## Experiment

#TODO: description

In [ ]:
from tokenizer import WordTokenizer

min_frequency = 1
tokenizer = WordTokenizer().train(ds.train["text"].tolist(), vocab_size=None, min_frequency=min_frequency)
token_indicator_encoder = rs.TokenIndicator(tokenizer)
bag_of_words_encoder = rs.BagOfWords(tokenizer)
tfidf_encoder = rs.Tfidf(tokenizer).fit(ds.train["text"].tolist())

my_seeds = [0]
num_folds = 5
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(ds.train["text"], ds.train["label"])), 
            k=num_folds,
            seed=seed)
    )

results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)

emotions_outcome_space = part1.clslib.NamedClasses(names=ds.emotions)

In [16]:
tokenizer.save(f"{results_folder}/word_tokenizer_vs_{len(tokenizer.get_vocab())}_mf_{min_frequency}.json")

In [ ]:
configs = {
    'logistic_indicator-enc': {
        'encoder': token_indicator_encoder,
        'model_option': 'logistic',
        'extra_args':{
            'loglin_args': {
                'max_iter': 300,
            }
        }
    },
    'mlp_indicator-enc': {
        'encoder': token_indicator_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 1,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
    'deeper_mlp_indicator-enc': {
        'encoder': token_indicator_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 2,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
    'logistic_bag-of-words-enc': {
        'encoder': bag_of_words_encoder,
        'model_option': 'logistic',
        'extra_args':{
            'loglin_args': {
                'max_iter': 300,
            }
        }
    },
    'mlp_bag-of-words-enc': {
        'encoder': bag_of_words_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 1,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
    'deeper_mlp_bag-of-words-enc': {
        'encoder': bag_of_words_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 2,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
    'logistic_tfidf-enc': {
        'encoder': tfidf_encoder,
        'model_option': 'logistic',
        'extra_args':{
            'loglin_args': {
                'max_iter': 300,
            }
        }
    },
    'mlp_tfidf-enc': {
        'encoder': tfidf_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 1,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
    'deeper_mlp_tfidf-enc': {
        'encoder': tfidf_encoder,
        'model_option': 'mlp',
        'extra_args':{
            'smlp_args': {
                'hidden_dim': 128, 
                'num_hidden_layers': 2,
                'dropout': 0.1, 
                'batch_size': 512, 
                'learning_rate': 1e-3, 
                'weight_decay': 1e-4, 
                'num_epochs': 10,
                'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            }
        }
    },
}

my_classifiers = OrderedDict()
for config in configs:
    my_classifiers[config] = []
    for run, seed in enumerate(my_seeds):
        run_folder = f"{results_folder}/{config}/seed-{seed}"
        my_classifiers[config].append(
            part1.experiment.cross_validation_experiment(
                data_splits=my_cross_validation_sets[run],
                outcome_space=emotions_outcome_space,
                encoder=configs[config]['encoder'],
                model_option=configs[config]['model_option'],
                save_to=run_folder,
                **configs[config]['extra_args'],
            )
        )

In [18]:
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))